# aggregate_baseline_vs_kd — Perbandingan Per-Fold Student Baseline vs Student KD v3 (5-Fold)

Jalankan **setelah kelima fold baseline DAN kelima fold KD selesai dievaluasi** (boleh sebagian — fold yang belum lengkap di salah satu model dikeluarkan dari perbandingan berpasangan dan hanya jadi warning):

1. **Add Input**: output notebook eval k-fold baseline (berisi `student_baseline_overall_fold{N}_9class.csv`) dan output notebook eval k-fold KD (berisi `student_kd_overall_fold{N}_9class.csv`) — total 10 CSV, boleh dari beberapa Input. Tidak perlu dikumpulkan manual — CSV dicari otomatis rekursif di `/kaggle/input`.
2. **Run All** (CPU cukup, hitungan detik).
3. Hasil: `kfold_baseline_vs_kd_summary.csv` di panel Output + tabel markdown di bawah cell (siap tempel): baseline, KD, dan selisih KD − baseline per fold + mean ± SD, serta perbandingan berpasangan.

**Catatan statistik.** Pasangan = fold yang sama (`folds.json` sama → test/val identik, train seed `42+FOLD` identik). Dengan n = 5 (df = 4) uji-t berpasangan berdaya rendah, dan kelima fold tidak sepenuhnya independen (train disubsample dari pool train yang sama), jadi p-value hanya **indikatif** — laporkan apa adanya. Uji hanya dijalankan untuk mAP@0.5 malam dan penurunan relatif; metrik lain deskriptif. SD = sampel (n − 1); SD pada kolom selisih adalah SD dari selisih per-fold.


In [ ]:
# =============== KONFIGURASI ===============
# Folder yang di-scan untuk mencari CSV overall per fold.
# Di Kaggle: add output notebook eval baseline & KD (kelima fold) sebagai Input, biarkan default.
SEARCH_DIRS = ["/kaggle/input", "/kaggle/working"]
OUT_PATH    = "/kaggle/working/kfold_baseline_vs_kd_summary.csv"
# ===========================================

import csv
import os

K = 5
PATTERNS = {
    "baseline": "student_baseline_overall_fold{fold}_9class.csv",
    "kd":       "student_kd_overall_fold{fold}_9class.csv",
}


def find_csv(kind, fold):
    """Cari CSV overall fold N untuk `kind` di mana pun dalam SEARCH_DIRS (rekursif)."""
    target = PATTERNS[kind].format(fold=fold)
    hits = []
    for base in SEARCH_DIRS:
        if not os.path.isdir(base):
            continue
        for root, _dirs, files in os.walk(base):
            if target in files:
                hits.append(os.path.join(root, target))
    if len(hits) > 1:
        print(f"[WARNING] {target} ditemukan {len(hits)}x, memakai yang pertama: {hits[0]}")
    return hits[0] if hits else None


def read_fold(path):
    rows = {}
    with open(path) as f:
        for r in csv.DictReader(f):
            if r["split"] in rows:
                print(f"[WARNING] {path}: split '{r['split']}' muncul lebih dari sekali -- memakai baris terakhir")
            rows[r["split"]] = r
    assert "daytime" in rows and "night" in rows, f"{path}: butuh baris daytime & night"
    d, n = rows["daytime"], rows["night"]
    m50d, m50n = float(d["map50"]), float(n["map50"])
    return {
        "map50_day": m50d, "map50_night": m50n,
        "map5095_day": float(d["map50_95"]), "map5095_night": float(n["map50_95"]),
        # Eq. (6) paper, dalam persen: (mAP50_siang - mAP50_malam) / mAP50_siang x 100
        "rel_drop": (m50d - m50n) / m50d * 100 if m50d else float("nan"),
    }


data = {"baseline": {}, "kd": {}}
for kind in data:
    for fi in range(K):
        p = find_csv(kind, fi)
        if p:
            data[kind][fi] = read_fold(p)
            print(f"[INFO] {kind:8s} fold_{fi}: {p}")
        else:
            print(f"[WARNING] {kind} fold_{fi}: {PATTERNS[kind].format(fold=fi)} tidak ditemukan")

folds = sorted(set(data["baseline"]) & set(data["kd"]))
only_one = sorted(set(data["baseline"]) ^ set(data["kd"]))
if only_one:
    print(f"[WARNING] fold {only_one} hanya punya salah satu model -- dikeluarkan dari perbandingan berpasangan.")
assert folds, "Tidak ada fold yang punya CSV baseline DAN KD. Cek Input notebook."
print(f"\n[INFO] Fold berpasangan: {folds} (n={len(folds)})")


In [ ]:
import statistics
from scipy import stats

# (kunci, label, lebih-besar-lebih-baik, jumlah desimal)
METRICS = [
    ("map50_day",     "mAP@0.5 siang",         True,  4),
    ("map50_night",   "mAP@0.5 malam",         True,  4),
    ("map5095_day",   "mAP@0.5:0.95 siang",    True,  4),
    ("map5095_night", "mAP@0.5:0.95 malam",    True,  4),
    ("rel_drop",      "penurunan relatif (%)", False, 2),   # lebih kecil = lebih baik
]
T_TEST_ON = ("map50_night", "rel_drop")   # sesuai rencana revisi; metrik lain hanya deskriptif
n = len(folds)


def mean_sd(vals):
    m = sum(vals) / len(vals)
    sd = statistics.stdev(vals) if len(vals) > 1 else float("nan")   # SD sampel (n-1)
    return m, sd


def fmt(x, nd, sign=False):
    if x != x:
        return "n/a"
    return f"{x:+.{nd}f}" if sign else f"{x:.{nd}f}"


vals = {kind: {m: [data[kind][fi][m] for fi in folds] for m, *_ in METRICS} for kind in data}
delta = {m: [k - b for k, b in zip(vals["kd"][m], vals["baseline"][m])] for m, *_ in METRICS}

summ = {}
for m, label, higher_better, nd in METRICS:
    b_m, b_sd = mean_sd(vals["baseline"][m])
    k_m, k_sd = mean_sd(vals["kd"][m])
    d_m, d_sd = mean_sd(delta[m])
    wins = sum((d > 0) if higher_better else (d < 0) for d in delta[m])
    t_stat = p_val = float("nan")
    if m in T_TEST_ON and n >= 2 and d_sd > 0:        # d_sd == 0 / nan -> uji-t tidak terdefinisi
        res = stats.ttest_rel(vals["kd"][m], vals["baseline"][m])   # dua sisi; t > 0 berarti KD > baseline
        t_stat, p_val = float(res.statistic), float(res.pvalue)
    summ[m] = dict(b=(b_m, b_sd), k=(k_m, k_sd), d=(d_m, d_sd), wins=wins, t=t_stat, p=p_val)

# ---------------- CSV ----------------
cols = ["fold"] + [f"{m}_{s}" for m, *_ in METRICS for s in ("baseline", "kd", "delta")]
with open(OUT_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(cols)
    for i, fi in enumerate(folds):
        row = [f"fold_{fi}"]
        for m, *_ in METRICS:
            row += [f"{vals['baseline'][m][i]:.4f}", f"{vals['kd'][m][i]:.4f}", f"{delta[m][i]:.4f}"]
        w.writerow(row)
    for stat_name, idx in (("mean", 0), ("sd", 1)):
        row = [stat_name]
        for m, *_ in METRICS:
            row += [fmt(summ[m]["b"][idx], 4), fmt(summ[m]["k"][idx], 4), fmt(summ[m]["d"][idx], 4)]
        w.writerow(row)
    # baris statistik berpasangan: hanya kolom *_delta yang terisi
    for name, key, nd in (("n_paired_folds", None, 0), ("kd_better_folds", "wins", 0),
                          ("paired_t_statistic", "t", 4), ("paired_t_pvalue", "p", 4)):
        row = [name]
        for m, *_ in METRICS:
            if key is None:
                cell = str(n)
            else:
                v = summ[m][key]
                cell = "" if (key in ("t", "p") and v != v) else (str(v) if key == "wins" else f"{v:.{nd}f}")
            row += ["", "", cell]
        w.writerow(row)
print(f"[INFO] Tersimpan: {OUT_PATH}\n")

# ---------------- tabel markdown ----------------
HDR = "| fold | " + " | ".join(label for _, label, _, _ in METRICS) + " |"
SEP = "|---|" + "---:|" * len(METRICS)


def per_fold_table(title, getter, mean_getter, sign=False):
    print(f"**{title}**\n")
    print(HDR)
    print(SEP)
    for i, fi in enumerate(folds):
        print(f"| fold_{fi} | " + " | ".join(fmt(getter(m, i), nd, sign) for m, _, _, nd in METRICS) + " |")
    print("| **mean ± SD** | " + " | ".join(
        f"**{fmt(mean_getter(m)[0], nd, sign)} ± {fmt(mean_getter(m)[1], nd)}**" for m, _, _, nd in METRICS) + " |")
    print()


per_fold_table("Student Baseline", lambda m, i: vals["baseline"][m][i], lambda m: summ[m]["b"])
per_fold_table("Student KD v3", lambda m, i: vals["kd"][m][i], lambda m: summ[m]["k"])
per_fold_table("Selisih KD − baseline (penurunan relatif dalam poin persentase; negatif = KD lebih baik)",
               lambda m, i: delta[m][i], lambda m: summ[m]["d"], sign=True)

print("**Perbandingan berpasangan (fold yang sama = pasangan)**\n")
print("| metrik | Δ KD − baseline (rata-rata ± SD) | fold KD lebih baik | t | p-value (paired t-test, dua sisi) |")
print("|---|---:|---:|---:|---:|")
for m, label, higher_better, nd in METRICS:
    s = summ[m]
    tested = m in T_TEST_ON
    t_txt = fmt(s["t"], 3) if tested else "—"
    p_txt = fmt(s["p"], 4) if tested else "—"
    print(f"| {label} | {fmt(s['d'][0], nd, True)} ± {fmt(s['d'][1], nd)} | {s['wins']}/{n} | {t_txt} | {p_txt} |")
print(f"\n> n = {n} fold berpasangan (df = {n - 1}). p-value dilaporkan apa adanya dan hanya indikatif: "
      "n kecil dan fold tidak sepenuhnya independen. \"Fold KD lebih baik\": mAP lebih tinggi / penurunan relatif lebih rendah. "
      "Metrik tanpa uji (—) hanya deskriptif.")
